# **Structural visualization and alignment**

In this exercise we will introduce molecular visualizations.

It is intended to help you achieve the following learning objectives:
* Visualize molecular structures with different representations. Compare the advantages and disadvantages of several representations.
* Align coordinates of protein structures with different amino acid sequences.

Assuming that you are reading this Notebook on Github, you will need to download it to your account on SDSC Expanse. To do that, log onto the Expanse User Portal, select *Shell*, and paste the following commands into the terminal:

```bash
mkdir -p ~/exercises
cd ~/exercises
curl -L -o 02-Structural_Visualization.ipynb https://raw.githubusercontent.com/daveminh/Chem456-2026S/refs/heads/main/exercises/02-Structural_Visualization.ipynb
```

To start JupyterLab, you can paste the following commands into the shell. You can adjust the time limit, but 1.5 hours is good to run for the class period and does not use too many computing resources in case you forget to shut down.

```bash
/cm/shared/apps/sdsc/galyleo/galyleo launch --account iit130 --partition shared --cpus 4 --memory 8 --time-limit 01:30:00 --interface lab --conda-env visualization --conda-init "$HOME/miniconda3/etc/profile.d/conda.sh"
```

After you start JupyterLab, navigate to the notebook.

The exercise will be graded based on submitting your answers to the questions after ```-->``` on Canvas.

# **Part 1 - Visualizing molecular structures with nglview**

In this exercise, we will visualize the structures of various biomolecules using [nglview](https://github.com/nglviewer/nglview), which allows for molecular visualization in Jupyter notebooks.

This exercise is an abbreviated and adapted version of [Lab 02 of IIBM3202 Molecular Modeling and Simulation](https://github.com/pb3lab/ibm3202/blob/master/tutorials/lab02_molviz.ipynb) from the Institute for Biological and Engineering at Pontificia Universidad Catolica de Chile.

## Retrieving structures

Now let us look at the structure of an important SARS-CoV-2 drug target, the main protease (MPro). Many structures of this enzyme are available on the [Protein Data Bank](https://www.rcsb.org/), including [5rh2](https://www.rcsb.org/structure/5rh2). This particular structure [helped inspire a clinical candidate](https://doi.org/10.1101/2022.01.26.477782) for COVID-19 treatment.

<p align = "justify">You can retrieve PDB structures from its website (www.rcsb.org). You can also directly use the terminal to download a given PDB file with known accession code as shown below, where XXXX must be replaced by the replaced by the 4-letter PDB code:

```
!wget http://www.rcsb.org/pdb/files/XXXX.pdb.gz
!gunzip XXXX.pdb.gz
```

Or you can use prody.

In [ ]:
import prody
PDB_5rh2 = prody.parsePDB('5rh2')
!gunzip 5rh2.pdb.gz

In [ ]:
# To keep things organized, move the file into a special directory for this exercise
!mkdir -p 02
!mv 5rh2.pdb 02/

## Creating and modifying an NGLWidget

The following will create a default visualization of the PDB structure as a NGLWidget. The protein is shown as a cartoon colored by `residueIndex`

Try some mouse controls:

* `drag-left` rotate scene
* `drag-middle` pan/translate scene
* `scroll` zoom scene
* `hoverPick` show tooltip for hovered component element
* `clickPick` middle auto view picked component element

You probably want to use `View -> Full Screen`.

In [ ]:
import nglview as nv
view = nv.show_prody(PDB_5rh2)
view.display(gui=True) # You can turn the gui on or off. By default, it is off.

The [NGLWidget](https://nglviewer.org/nglview/latest/api.html#nglview.NGLWidget) object can be modified in many ways by combining representations based on different selections and colors:
* [representation](https://nglviewer.org/ngl/api/manual/molecular-representations.html) - ways to show parts of the system. Key representations include `cartoon`, `ball+stick`, `licorice`, `surface`, and `label`.
* [selection](https://nglviewer.org/ngl/api/manual/selection-language.html) - which parts of the structure to show. The selection language includes keywords (e.g. `protein`, `backbone`, `sidechain`, `water`), expressions, and logical operators (`and`, `or`), and makes use of parentheses.
* [color](https://nglviewer.org/ngl/api/manual/coloring.html) - what color/color scheme should be used.

The code snippets below will programmatically (opposed to through the GUI) change the NGLWidget above. This is helpful for reproducibility and for scripting. Try various combinations and see what they do. You may need to clear old representations in order to see a new one.

In [ ]:
view.clear_representations() # To start all over

In [ ]:
view.add_representation('ball+stick')

In [ ]:
view.add_representation('licorice', selection="backbone") # Show the protein backbone in licorice

In [ ]:
view.add_representation('cartoon', selection="protein", color="purple") # Show the protein as a purple cartoon

In [ ]:
# Adds a molecular surface color by (approximate) electrostatics
view.add_representation('surface', selection="protein", opacity=0.4, colorScheme="electrostatic")

### --> Write code to show water in a ball+stick representation

## Primary structure

Now let's look at a small part of the protein backbone. Besides selecting part of a protein in NGLView, we can create NGLWidget using part of a protein in the [prody selection language](http://www.bahargroup.org/prody/manual/reference/atomic/select.html).

In [ ]:
view = nv.show_prody(PDB_5rh2.select("resnum 227 to 237"))
view.clear_representations() # To start all over
view.add_representation("ball+stick")
view.display(gui=True)

### --> Modify `view.add_representation("ball+stick")` to only show the backbone. Hint: add a selection string.

### --> What are the three-letter codes for amino acids 227 to 231? Hint: hover your mouse along the structure.

## Secondary structure

This part of the protein is part of an alpha helix. The following code shows some hydrogen bonds along the backbone. Note the hydrogen itself is not visible in most crystal structures.

In [ ]:
view.add_distance(atom_pair = [["227.O","231.N"]], color='green', labelColor='black')
view.add_distance(atom_pair = [["228.O","232.N"]], color='green', labelColor='black')
view.add_distance(atom_pair = [["229.O","233.N"]], color='green', labelColor='black')

### --> Which residue number has a hydrogen bond with the backbone nitrogen of residue 235?

In [ ]:
view.add_representation("cartoon") # Adds a cartoon representation of the alpha helix

Now let's look at a part of the protein with a different type of secondary structure: a beta sheet.

In [ ]:
view = nv.show_prody(PDB_5rh2.select("resnum 19 to 30"))
view.clear_representations() # To start all over
view.add_representation("ball+stick") # This only shows one asymmetric unit opposed to the biological assembly
view.add_distance(atom_pair = [["20.O","27.N"]], color='green')
view.display()

### --> Which residue index has a hydrogen bond with the backbone nitrogen of residue 22?

In [ ]:
view.add_representation("cartoon") # Adds a cartoon representation to the beta sheet

## Tertiary structure

In [ ]:
sel = PDB_5rh2.select('(same residue as within 8 of resnum 140) and not water')
view = nv.show_prody(sel)
view.add_representation("ball+stick")
view.display()

### --> Which residue (three-letter code and number) has a side chain that stacks with the side chain of PHE 140?

## Quaternary structure

While we have focused on polypeptide chain, MPro is much more active as a homodimer.  Let's look at both sides. By default, NGLView will show a "Biological Assembly", which for MPro applies translation and operations in a PDB file to show the dimer.

In [ ]:
view = nv.show_structure_file("02/5rh2.pdb")
view.display()

## Ligand interactions

Let's focus on where the protein holds the ligand. This ligand is a competitive inhibitor that binds to the catalytic site. Try adding different combinations of representations.

In [ ]:
sel = PDB_5rh2.select('(same residue as within 12 of resname UH7) and not water')
view = nv.show_prody(sel)
view.display()

In [ ]:
view.clear_representations() # To start all over

In [ ]:
view.add_representation('licorice', selection="UH7") # Show the ligand in licorice

In [ ]:
view.add_representation('ball+stick', selection="sidechain") # Show the side chains in ball+stick

In [ ]:
view.add_representation('ball+stick', selection="(41 or 145) and (sidechain or .CA)") # Show the catalytic dyad in ball+stick

In [ ]:
# Adds a molecular surface (artificial because the protein is truncated)
view.add_representation('surface', selection="protein", opacity=0.4)

In [ ]:
# Add labels to the catalytic dyad
view.add_representation('label', selection="41.CB", color='green')
view.add_representation('label', selection="145.CB", color='green')

In [ ]:
# Add a label for the distance between key atoms in the catalytic dyad
view.add_distance(atom_pair = [["41.NE2","145.SG"]], color='green')

### --> Write code to add a label for the distance between the sulfur of cysteine 145 and the carbonyl carbon in the amide of the ligand (Hint: hover over the atom to find its index.)

### --> Describe at least one advantage and disadvantage of each of the following representations:
* `ball+stick`
* `licorice`
* `cartoon`


# **Part 2 - Structural alignment with prody**

It has been observed that **structure space is rather small when compared with the sequence space**. In fact, hierarchical structure classifications such as **CATH** have demonstrated that only a few proteins of the enormous pool of structures being deposited each year in the Protein Data Bank are known to provide novel folds. In this regard, it seems that we have discovered almost all protein folds. Moreover, the elegant experiment by Chothia & Lesk [Chothia C & Lesk AM (1986) _EMBO J_ 5(4), 823–826] revealed that, generally, **protein structures are quite similar between proteins when their sequence identity is > 30-40%**, with some remarkable exceptions to this rule).

As an illustrative example of the fact that similar sequences often lead to similar structure, visualize crystal structures of MPro from the original SARS-CoV and SARS-CoV-2.

In [ ]:
PDB_1wof = prody.parsePDB('1wof')

view = nv.show_prody(PDB_1wof.select('chain A')) # 1wof has two chains in the PDB file, so we select one.
view.add_component(PDB_5rh2)
view

Wait, the proteins are in different positions! How can we compare them? We will need to use prody to superpose the structures. Here we use prody to do so.

In [ ]:
# Define a mapping between protein chains
map_from_5rh2_to_1wof, map_from_1wof_to_5rh2, seqid, overlap = \
  prody.matchChains(PDB_5rh2.select('protein'),PDB_1wof.select('protein'))[0] # There are two mappings, but just keep one
transformation = prody.calcTransformation(map_from_1wof_to_5rh2, map_from_5rh2_to_1wof) # Calculate a transformation that superposes the chains
prody.applyTransformation(transformation, PDB_1wof) # Apply the transformation

Now we can see the structures on top of each other.

In [ ]:
view = nv.show_prody(PDB_1wof.select("chain A")) # It is also possible to select only chain A
view.add_component(PDB_5rh2)
view.display()

While the tertiary structures is quite similar, there can be differences in the side chain positions. I designed the visualization below to highlight these differences.

In [ ]:
view = nv.NGLWidget()
view.add_component(PDB_1wof)
view[0].clear_representations()
view[0].add_representation("cartoon", selection=":A", color="yellow")
# Color side chains carbon blue
view[0].add_representation("ball+stick", selection="sidechainAttached and :A", color="yellow")
view[0].add_representation("ball+stick", selection="sidechainAttached and (not _C) and :A")
view[0].add_representation("cartoon", selection=":B", color="grey", opacity=0.5)

view.add_component(PDB_5rh2)
view[1].clear_representations()
view[1].add_representation("cartoon", selection=":A", color="purple")
view[1].add_representation("ball+stick", selection="sidechainAttached and :A", color="purple")
view[1].add_representation("ball+stick", selection="sidechainAttached and (not _C) and :A")
view.display(gui=True)

### --> Look more closely at the side chain positions. Do you notice any patterns with how positions of the side chains are the same or different
* on the surface
* in the interior of the protein
* at the interface between subunits?

# **Ending the session**

To end the JupyterLab session, you should click on `File->Shut Down`. Closing your browser tab does **not** necessarily stop the job. You can find your running jobs with `squeue -u $USER` and cancel a job with `scancel <JOBID>` (replace `<JOBID>`).

When you are done with this exercise, save the file in ~/exercises. Submit your answers to the questions after ```-->``` on Canvas.